<a href="https://colab.research.google.com/github/Ali-Shahrez/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

**Contract, in plain words:**

1. **Unit of analysis (one row):** one pseudonymized **content item**, rolled up over one calendar month. The warehouse's native grain is `report_date × client_hash_id × content_hash_id` (one row per page per day) in `fact_content_daily_performance`; I aggregate that daily grain up to one row per content item for the month I'm studying.
2. **Table(s):** `fact_content_daily_performance`, filtered to the `month=2026-03` partition. (I check its actual columns below with `DESCRIBE` instead of assuming names.)
3. **Time window:** a single **mid-panel month, March 2026**, split internally into two 15-day halves — `2026-03-01`→`2026-03-15` ("prev") and `2026-03-16`→`2026-03-31` ("last") — the same trick `trend_direction` uses in the starter CSV, just rebuilt here on real warehouse rows.
4. **What I'd predict/rank (label or proxy):** a **proxy label** — did this page's impressions drop in the second half of March vs. the first half (mirrors `trend_direction == "down"`). This is explicitly the same-window proxy I flagged as a limitation in ML-03, not the capstone's real target. The real capstone target will be a genuine future-window label (March → April), which needs a second month partition and is out of scope for this exercise.
5. **One thing deliberately excluded:** `dim_content`'s keyword/content-metadata fields (`word_count`, `content_type`, `search_volume`, etc.). This week is about proving the daily-fact table's own grain, windows, and availability flag — pulling in content metadata now would be scope creep before the label design is settled, and I'd rather add it once ML-05/06 need it.


In [39]:
%pip -q install duckdb
from google.colab import userdata

HF_TOKEN = userdata.get('HF_TOKEN')

import duckdb
con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
TABLES = {
    'dim_clients':       f"read_parquet('{REL}/dim_clients.parquet')",
    'dim_content':       f"read_parquet('{REL}/dim_content.parquet')",
    'fact_daily':        f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
    'fact_daily_march':  f"read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')",
}


print(con.sql(f"DESCRIBE SELECT * FROM {TABLES['fact_daily_march']}").df())


                 column_name column_type null   key default extra
0                report_date        DATE  YES  None    None  None
1             client_hash_id     VARCHAR  YES  None    None  None
2            content_hash_id     VARCHAR  YES  None    None  None
3             client_has_gsc     BOOLEAN  YES  None    None  None
4             client_has_ga4     BOOLEAN  YES  None    None  None
5         gsc_data_available     BOOLEAN  YES  None    None  None
6         ga4_data_available     BOOLEAN  YES  None    None  None
7            gsc_impressions      BIGINT  YES  None    None  None
8                 gsc_clicks      BIGINT  YES  None    None  None
9           gsc_sum_position      BIGINT  YES  None    None  None
10          gsc_avg_position      DOUBLE  YES  None    None  None
11             ga4_pageviews      BIGINT  YES  None    None  None
12              ga4_sessions      BIGINT  YES  None    None  None
13                 ga4_users      BIGINT  YES  None    None  None
14      ga

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

| Field | Bucket | Why |
|---|---|---|
| `content_hash_id` | Context | Grouping/joining only — never a feature (pseudonym, no real signal) |
| `client_hash_id` | Context | Grouping + client-holdout splits only — never a feature |
| `report_date` | Context | Used to build the prev/last 15-day windows, not fed to a model directly |
| `gsc_impressions` (prev-15d sum) | Feature | Knowable at the decision moment — occurred entirely before the label window opens |
| `gsc_clicks` (prev-15d sum) | Feature | Same — prior-window observed signal |
| `gsc_avg_position` (prev-15d avg) | Feature | Same — prior-window observed signal |
| days with impressions (prev-15d count) | Feature | Same — prior-window observed signal |
| `ga4_sessions` (prev-15d sum, availability-filtered) | Feature | Prior-window signal, but only valid where `ga4_data_available IS TRUE` |
| `ga4_data_available` | Context | A filter flag, not a feature — tells me whether GA4 zeros are real or "not tracked yet" |
| `gsc_impressions` (last-15d sum) | Label / proxy source | This is what the decline label is computed FROM — never a feature (the trap in section 3 proves why) |
| `dim_content.*` (word_count, content_type, search_volume, ...) | Excluded | Out of scope this week — see section 1, item 5 |
| Any FlyRank product flag (`health_score`, `priority_score`, `action_type`) | Excluded | Not shipped in this release at all; if I ever rebuild one, it's a baseline to beat, never a feature |


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

**Query 1 — grain.** One row really is one `report_date × client_hash_id × content_hash_id` in the daily fact. Zero rows back proves the grain holds.


In [40]:
tbl = TABLES['fact_daily_march']

grain_probe = con.sql(f"""
    SELECT report_date, client_hash_id, content_hash_id, COUNT(*) AS c
    FROM {tbl}
    GROUP BY 1, 2, 3
    HAVING COUNT(*) > 1
    LIMIT 5
""").df()

print(f'duplicate (date, client, content) rows in month=2026-03: {len(grain_probe)}')
grain_probe

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

duplicate (date, client, content) rows in month=2026-03: 0


,report_date,client_hash_id,content_hash_id,c


**Query 2 — counts + date span.** Row count and date range inside the `month=2026-03` partition, plus how many distinct clients/content items show up that month.


In [41]:
span = con.sql(f"""
    SELECT COUNT(*) AS n_rows,
           MIN(report_date) AS min_date,
           MAX(report_date) AS max_date,
           COUNT(DISTINCT client_hash_id) AS n_clients,
           COUNT(DISTINCT content_hash_id) AS n_content_items
    FROM {tbl}
""".format(tbl=TABLES['fact_daily_march'])).df()

span


,n_rows,min_date,max_date,n_clients,n_content_items
0,9841378,2026-03-01,2026-03-31,55,331437


**Query 3 — availability.** Filter with `IS TRUE` on `ga4_data_available` and show how many rows survive vs. the total — the doc's warning is that zeros before a client's GA4 start are "not tracked," not "no engagement," so I never trust a GA4 zero without checking this flag first.


In [42]:
availability = con.sql(f"""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(*) FILTER (WHERE ga4_data_available IS TRUE) AS ga4_available_rows,
        ROUND(100.0 * COUNT(*) FILTER (WHERE ga4_data_available IS TRUE) / COUNT(*), 1) AS pct_available
    FROM {tbl}
""".format(tbl=TABLES['fact_daily_march'])).df()

availability


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,ga4_available_rows,pct_available
0,9841378,413966,4.2


**Supplementary verification (beyond the required three) — following up on Query 3's result.**

Query 3 showed only 4.2% GA4 coverage. Before trusting that number, I ran a few follow-up checks: the same availability check for GSC (for comparison), the raw TRUE/FALSE/NULL breakdown for both flags, whether coverage gaps are random or client-driven, and how the availability flags relate to each client's `client_has_gsc`/`client_has_ga4` status. These are my own investigation, not the three required queries — Queries 1–3 above already satisfy that requirement on their own.

In [43]:
gsc_availability = con.sql(f"""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(*) FILTER (WHERE gsc_data_available IS TRUE) AS gsc_available_rows,
        ROUND(100.0 * COUNT(*) FILTER (WHERE gsc_data_available IS TRUE) / COUNT(*), 1) AS pct_available
    FROM {tbl}
""").df()

gsc_availability

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,gsc_available_rows,pct_available
0,9841378,3611061,36.7


**Raw breakdown check.** `IS TRUE` in Query 3 collapses `FALSE` and `NULL` together — this shows them separately, for `ga4_data_available` first.

In [44]:
con.sql(f"SELECT ga4_data_available, COUNT(*) FROM {tbl} GROUP BY 1").df()

,ga4_data_available,count_star()
0,<NA>,3018741
1,False,6408671
2,True,413966


**Same raw breakdown, for `gsc_data_available`.** Notably: no `NULL` row appears here at all (see markdown below) — a first hint that GSC and GA4 fail differently, not just at different rates.

In [45]:
con.sql(f"SELECT gsc_data_available, COUNT(*) FROM {tbl} GROUP BY 1").df()

,gsc_data_available,count_star()
0,False,6230317
1,True,3611061


**Is the gap client-driven or random?** If GA4 coverage jumps cleanly between 0 and 31 (or partial values consistent with a single start date), that points to onboarding timing, not a data-quality problem. Counting *distinct dates* per client (not rows) tests this directly.

In [46]:
con.sql(f"""
    SELECT client_hash_id,
           COUNT(DISTINCT report_date) AS distinct_dates,
           COUNT(DISTINCT report_date) FILTER (WHERE ga4_data_available IS TRUE) AS distinct_dates_ga4_true
    FROM {tbl}
    GROUP BY 1
    ORDER BY distinct_dates DESC
    LIMIT 15
""").df()

,client_hash_id,distinct_dates,distinct_dates_ga4_true
0,client_62f4a7e64f5e0096,31,0
1,client_9958f0a7ae1df715,31,31
2,client_73cda7b4e4f265ea,31,8
3,client_fef1a8f436438636,31,26
4,client_08a6a72ff48e62c0,31,0
5,client_2c32078d69f2cbad,31,0
6,client_19b89ee4fe3db6da,31,2
7,client_ba65e80a1116ae41,31,31
8,client_3ffa76342f366962,31,21
9,client_f623b01661d4bfe4,31,31


**Cross-check: does `ga4_data_available` line up with `client_has_ga4`?** Testing whether `NULL` specifically means "this client never had GA4 at all," as opposed to some other kind of gap.

In [47]:
con.sql(f"""
    SELECT ga4_data_available, client_has_ga4, COUNT(*)
    FROM {tbl}
    GROUP BY 1, 2
    ORDER BY 1, 2
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,ga4_data_available,client_has_ga4,count_star()
0,False,True,6408671
1,True,True,413966
2,<NA>,False,3018741


**Does an `IS NULL` check on a metric column catch the same gap?** Spoiler: no — see the write-up below the result. Unavailable values are zero-filled, not left as SQL `NULL`, so this check can't see what the availability flags already showed. Kept in the notebook because the negative result is itself the finding.

In [48]:
con.sql(f"""
    SELECT client_hash_id,
           AVG(CASE WHEN gsc_impressions IS NULL THEN 1.0 ELSE 0 END) AS pct_null_impressions
    FROM {tbl}
    GROUP BY 1
    ORDER BY 2 DESC
    LIMIT 10
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,client_hash_id,pct_null_impressions
0,client_73cda7b4e4f265ea,0.0
1,client_c182d11e4862a37d,0.0
2,client_f623b01661d4bfe4,0.0
3,client_8ae2bfb5aa1ffa1e,0.0
4,client_a2eeb8899886adde,0.0
5,client_d211cb07b9059bab,0.0
6,client_4a18d1793d92fb84,0.0
7,client_62f4a7e64f5e0096,0.0
8,client_fef1a8f436438636,0.0
9,client_ba65e80a1116ae41,0.0


**Same cross-check, for GSC — does `gsc_data_available` line up with `client_has_gsc`?**

In [49]:
con.sql(f"""
    SELECT gsc_data_available, client_has_gsc, COUNT(*)
    FROM {tbl}
    GROUP BY 1, 2
    ORDER BY 1, 2
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,gsc_data_available,client_has_gsc,count_star()
0,False,True,6230317
1,True,True,3611061


**Wrap-up of the supplementary checks:** GA4 coverage gaps are `NULL` exactly where `client_has_ga4 = False` (whole clients without GA4 at all), and partial `FALSE`/`TRUE` splits elsewhere consistent with tracking starting mid-month — not random noise. GSC has no `NULL` case at all (every client in this slice has `client_has_gsc = True`); its 36.7% vs. GA4's 4.2% reflects *when* tracking started, not *whether* it exists. And `IS NULL` checks on metric columns miss unavailable rows entirely, since they're zero-filled rather than nulled. This gives me real, checked confidence in Query 3's number, and in section 4's write-up below, rather than just accepting a surprising percentage at face value.


### Five features, max

All five come from the **prev-15d half of March only** (`2026-03-01`→`2026-03-15`) — the window the label (built from the last-15d half) has not happened yet at that point, so none of these can see their own answer.


In [50]:
feature_frame = con.sql(f"""
    SELECT
        content_hash_id,
        client_hash_id,
        SUM(gsc_impressions)                                        AS imp_prev15,
        SUM(gsc_clicks)                                              AS clicks_prev15,
        AVG(gsc_avg_position)                                        AS avg_position_prev15,
        COUNT(*) FILTER (WHERE gsc_impressions > 0)                  AS days_with_impressions_prev15,
        SUM(ga4_sessions) FILTER (WHERE ga4_data_available IS TRUE)  AS sessions_prev15
    FROM {tbl}
    WHERE report_date BETWEEN DATE '2026-03-01' AND DATE '2026-03-15'
    GROUP BY 1, 2
    HAVING imp_prev15 > 0
""").df()

print(f'{len(feature_frame):,} content items with prior-window impressions')
feature_frame.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

151,981 content items with prior-window impressions


,content_hash_id,client_hash_id,imp_prev15,clicks_prev15,avg_position_prev15,days_with_impressions_prev15,sessions_prev15
0,content_d0dff76c889de68f,client_62f4a7e64f5e0096,111.0,0.0,5.222776,13,NaN
1,content_ac8663da7484669a,client_62f4a7e64f5e0096,20.0,0.0,3.597222,9,NaN
2,content_39d7361b4945d504,client_62f4a7e64f5e0096,57.0,0.0,3.659683,15,NaN
3,content_d49a012dcb924e31,client_62f4a7e64f5e0096,246.0,0.0,4.520919,15,NaN
4,content_cec711b02f3bbde6,client_62f4a7e64f5e0096,199.0,2.0,4.086084,13,NaN


In [51]:
print(feature_frame['sessions_prev15'].notna().mean())

0.2610063099992762


**Feature notes — "knowable at the decision moment because...":**

1. `imp_prev15` (GSC impressions, Mar 1–15) — knowable because it's already-observed search activity that finished before the decision point (Mar 16).
2. `clicks_prev15` (GSC clicks, Mar 1–15) — same: fully observed before the decision point.
3. `avg_position_prev15` (mean GSC position, Mar 1–15) — same: a summary of already-completed search results, not a forecast.
4. `days_with_impressions_prev15` (days with ≥1 impression, Mar 1–15) — same: a count of days that have already happened.
5. `sessions_prev15` (GA4 sessions, Mar 1–15, availability-filtered) — knowable because it only uses rows where `ga4_data_available IS TRUE`, so it never mistakes "not tracked yet" for zero engagement, and it's still confined to the prior window.


### The trap — one label-derived column, on purpose

First, the honest label and a quick score using only the five prior-window features above. Then I add ONE column that the label is actually computed from (`imp_last15`, the second-half-of-March impressions) and watch the score jump toward perfect — this is notebook 02's leakage lesson, re-run here on real warehouse rows instead of the starter CSV.


In [52]:
labels = con.sql(f"""
    SELECT
        content_hash_id,
        SUM(gsc_impressions) AS imp_last15
    FROM {tbl}
    WHERE report_date BETWEEN DATE '2026-03-16' AND DATE '2026-03-31'
    GROUP BY 1
""").df()

data = feature_frame.merge(labels, on='content_hash_id', how='left')
data['imp_last15'] = data['imp_last15'].fillna(0)

# Same rule as trend_direction: down = last15 more than 20% below prev15.
data['is_declining'] = (data['imp_last15'] < 0.8 * data['imp_prev15']).astype(int)

print(f"base rate (share declining): {data['is_declining'].mean():.3f}")
data.head()

base rate (share declining): 0.327


,content_hash_id,client_hash_id,imp_prev15,clicks_prev15,avg_position_prev15,days_with_impressions_prev15,sessions_prev15,imp_last15,is_declining
0,content_d0dff76c889de68f,client_62f4a7e64f5e0096,111.0,0.0,5.222776,13,NaN,70.0,1
1,content_ac8663da7484669a,client_62f4a7e64f5e0096,20.0,0.0,3.597222,9,NaN,14.0,1
2,content_39d7361b4945d504,client_62f4a7e64f5e0096,57.0,0.0,3.659683,15,NaN,20.0,1
3,content_d49a012dcb924e31,client_62f4a7e64f5e0096,246.0,0.0,4.520919,15,NaN,83.0,1
4,content_cec711b02f3bbde6,client_62f4a7e64f5e0096,199.0,2.0,4.086084,13,NaN,403.0,0


In [53]:
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

honest_features = ['imp_prev15', 'clicks_prev15', 'avg_position_prev15',
                    'days_with_impressions_prev15', 'sessions_prev15']

model_data = data.dropna(subset=['is_declining']).copy()
model_data[honest_features] = model_data[honest_features].fillna(0)

X, y = model_data[honest_features], model_data['is_declining']
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)

honest_model = LogisticRegression(max_iter=1000).fit(X_tr, y_tr)
honest_auc = roc_auc_score(y_te, honest_model.predict_proba(X_te)[:, 1])

print(f'base rate (train): {y_tr.mean():.3f}')
print(f'HONEST score (5 prior-window features only), ROC AUC: {honest_auc:.3f}')

base rate (train): 0.327
HONEST score (5 prior-window features only), ROC AUC: 0.601


In [54]:
leaky_features = honest_features + ['imp_last15']
leak_data = data.dropna(subset=leaky_features + ['is_declining']).copy()
leak_data[leaky_features] = leak_data[leaky_features].fillna(0)

Xl, yl = leak_data[leaky_features], leak_data['is_declining']
Xl_tr, Xl_te, yl_tr, yl_te = train_test_split(Xl, yl, test_size=0.25, random_state=42, stratify=yl)

leaky_model = LogisticRegression(max_iter=1000).fit(Xl_tr, yl_tr)
leaky_auc = roc_auc_score(yl_te, leaky_model.predict_proba(Xl_te)[:, 1])

print(f'LEAKY score (adding imp_last15), ROC AUC: {leaky_auc:.3f}')
print(f'Jump: {honest_auc:.3f} -> {leaky_auc:.3f}')

LEAKY score (adding imp_last15), ROC AUC: 1.000
Jump: 0.601 -> 1.000


`imp_last15` isn't a real feature — it's the exact number `is_declining` is thresholded from
(`imp_last15 < 0.8 * imp_prev15`). Handing the model that column lets it reconstruct the label's
own formula instead of learning anything real, which is why AUC jumped from 0.601 to 1.000.
Deleted it; 0.601 is the number I'm keeping.

In [55]:
del leaky_features, leak_data, Xl, yl, Xl_tr, Xl_te, yl_tr, yl_te, leaky_model
print(f'Keeping the honest number: ROC AUC = {honest_auc:.3f} on {len(model_data):,} content items.')

Keeping the honest number: ROC AUC = 0.601 on 151,981 content items.


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

**This slice is thin, unevenly covered, and month-specific — none of the numbers above generalize automatically.**

1. **Unbalanced panel, concretely.** Only **55 of the warehouse's ~70 clients** have any rows in March 2026 at all (Query 2). The other ~15 simply aren't here — most likely their tracking (`gsc_data_start`/`ga4_data_start`) begins after March or ended before it. Every conclusion in this notebook describes March 2026 for these 55 clients specifically, not the warehouse as a whole, and won't automatically match another month.

2. **GA4 coverage is thin, and not thin the same way GSC is.** Only **4.2%** of March's ~9.8M page-day rows have `ga4_data_available = TRUE` (Query 3), versus **36.7%** for `gsc_data_available`. The two also fail differently: `ga4_data_available` is `NULL` for entire clients that never had GA4 connected (`client_has_ga4 = False`), while every client in this slice has `client_has_gsc = True` — GSC availability only varies by *when* each client's tracking started, not *whether* it exists at all. Concretely, this meant `sessions_prev15` (my GA4-based feature) came back non-null for only **26.1%** of content items in the feature frame — and even those non-null values are often built from a partial window (as few as 1 of 15 days), not a full 15-day sum. I kept the feature anyway, with this caveat attached, rather than pretend it's as solid as the GSC-based features.

3. **Zero-filled, not null — `IS NULL` checks are the wrong tool here.** Unavailable GA4/GSC values show up as real zeros in the metric columns, not as missing values; only the `*_data_available` flags reveal them. I confirmed this directly — an `IS NULL` check on `gsc_impressions` returned 0.0% missing everywhere, which looks reassuring but actually just means that check can't see the gap at all. Any future work on this table has to filter on the availability flags, never trust a bare `fillna(0)` or a null check to catch this.

4. **The label is a same-window proxy, not a real outcome — restated from section 1.** `is_declining` compares two halves of the *same* already-observed month. It answered a mechanical question ("did the second half already come in lower than the first"), not a predictive one ("will this page decline next month"). The leakage trap in section 3 exists precisely because this window design makes it easy to accidentally hand the model its own answer — a genuine future-window label (March features → April outcome) would still need care, but wouldn't have this specific same-table, same-query vulnerability, since the two windows would live in separate partitions.

5. **Row count doesn't match the documented average, and I haven't fully explained why.** March's ~9.8M rows are more than double the ~4.6M/month you'd get by spreading the warehouse's documented 78.8M rows evenly across its ~17-month span. This could simply reflect the panel growing denser over time (more clients/content onboarded later), but I haven't verified that — flagging it honestly rather than treating the row count as validated.

**Bottom line:** this contract is a proof of mechanics — grain, windows, availability, leakage — on one arbitrary, partially-covered month. It is not a validated basis for capstone-level claims about decline, and the real target (a future-window label spanning March→April or more) is the next step, not this one.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.